# Supervised Model Evaluation and Calibration (M2/M3/M5)

This notebook uses reusable repository code. Search and calibration decisions use training/validation folds only; the final test fixture is evaluated once for smoke evidence. Full ISOT/WELFake benchmark claims require raw datasets.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import learning_curve, validation_curve

from src.evaluation.metrics import (evaluate_predictions, mcnemar_test, paired_bootstrap_regression, regression_metrics)
from src.evaluation.plots import plot_learning_curve, plot_reliability_comparison, plot_roc_pr, plot_validation_curve
from src.features.text import TfidfTextPipeline
from src.models.classical import build_logistic_model, build_random_forest

train = pd.read_csv('../tests/fixtures/train.csv')
test = pd.read_csv('../tests/fixtures/test.csv')
tfidf = TfidfTextPipeline(min_df=1, max_df=1.0, max_features=200)
X_train = tfidf.fit_transform(train['content'])
X_test = tfidf.transform(test['content'])
y_train, y_test = train['label'].to_numpy(), test['label'].to_numpy()
baseline = build_logistic_model('l2', max_iter=500).fit(X_train, y_train)
forest = build_random_forest(n_estimators=20, random_state=42).fit(X_train.toarray(), y_train)
baseline_proba = baseline.predict_proba(X_test)
forest_proba = forest.predict_proba(X_test.toarray())
print(evaluate_predictions(y_test, baseline_proba).to_dict())


In [ ]:
report_dir = Path('../reports/evaluation')
report_dir.mkdir(parents=True, exist_ok=True)
plot_roc_pr(y_test, baseline_proba, report_dir / 'notebook_roc_pr.png', label='logistic')
plot_reliability_comparison(y_test, {'logistic': baseline_proba, 'forest': forest_proba}, report_dir / 'notebook_reliability.png')


In [ ]:
sizes, train_scores, validation_scores = learning_curve(
    build_logistic_model('l2', max_iter=500), X_train, y_train, cv=2, scoring='accuracy',
    train_sizes=np.asarray([0.5, 0.75, 1.0]),
)
plot_learning_curve(sizes, train_scores, validation_scores, report_dir / 'notebook_learning_curve.png')


In [ ]:
values, train_scores, validation_scores = validation_curve(
    build_logistic_model('l2', max_iter=500), X_train, y_train, param_name='classifier__C',
    param_range=[0.25, 1.0, 4.0], cv=2, scoring='accuracy',
)
plot_validation_curve(values, train_scores, validation_scores, report_dir / 'notebook_validation_curve.png', parameter_name='C')


In [ ]:
# Nested CV is implemented in src.evaluation.metrics; pass a fold-local search factory in a production run.
print('Nested-CV contract: nested_stratified_cross_validate')


In [ ]:
print(mcnemar_test(y_test, baseline_proba, forest_proba))
actual = np.asarray([1.0, 2.0, 3.0, 4.0])
pred_a, pred_b = np.asarray([1.1, 1.9, 3.2, 3.8]), np.asarray([1.5, 2.5, 2.5, 3.5])
print(regression_metrics(actual, pred_a))
print(paired_bootstrap_regression(actual, pred_a, pred_b, n_bootstrap=100, random_state=42))


In [ ]:
try:
    from src.models.classical import shap_values
    feature_names = np.asarray(tfidf.get_feature_names(), dtype=object)
    print(shap_values(forest, X_train.toarray(), feature_names, max_samples=100))
except RuntimeError as exc:
    print(f'SHAP optional dependency unavailable: {exc}')
